In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
import json

with open('circles.json') as f:
    data = json.load(f)

iterations = list(range(1, len(data) + 1))
lats = [d['lat'] for d in data]
lons = [d['lon'] for d in data]
radii = [d['radius_m'] for d in data]

delta_lat = [0] + [lats[i] - lats[i-1] for i in range(1, len(lats))]
delta_lon = [0] + [lons[i] - lons[i-1] for i in range(1, len(lons))]
delta_radius = [0] + [radii[i] - radii[i-1] for i in range(1, len(radii))]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Circle Movement & Shrinkage Over Time', fontsize=14)

axes[0, 0].plot(iterations, lats, color='steelblue')
axes[0, 0].set_title('Latitude')
axes[0, 0].set_ylabel('degrees')

axes[1, 0].plot(iterations, lons, color='darkorange')
axes[1, 0].set_title('Longitude')
axes[1, 0].set_ylabel('degrees')

axes[2, 0].plot(iterations, radii, color='seagreen')
axes[2, 0].set_title('Radius')
axes[2, 0].set_ylabel('meters')
axes[2, 0].set_xlabel('Iteration')

axes[0, 1].plot(iterations, delta_lat, color='steelblue')
axes[0, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[0, 1].set_title('Δ Latitude')
axes[0, 1].set_ylabel('degrees / step')

axes[1, 1].plot(iterations, delta_lon, color='darkorange')
axes[1, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[1, 1].set_title('Δ Longitude')
axes[1, 1].set_ylabel('degrees / step')

axes[2, 1].plot(iterations, delta_radius, color='seagreen')
axes[2, 1].axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[2, 1].set_title('Δ Radius')
axes[2, 1].set_ylabel('meters / step')
axes[2, 1].set_xlabel('Iteration')

for ax in axes.flat:
    ax.set_xlabel('Iteration')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from IPython.display import HTML

df = pd.DataFrame({
    'Iteration': iterations,
    'Latitude': lats,
    'Longitude': lons,
    'Radius (m)': radii,
    'Δ Latitude': delta_lat,
    'Δ Longitude': delta_lon,
    'Δ Radius (m)': delta_radius,
})
df.set_index('Iteration', inplace=True)

html = f'<div style="height:500px;overflow-y:auto;overflow-x:auto;">{df.to_html()}</div>'
display(HTML(html))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

start = 455  # iteration 456 (0-indexed)
iters_seg = np.array(iterations[start:])
lats_seg  = np.array(lats[start:])
lons_seg  = np.array(lons[start:])
radii_seg = np.array(radii[start:])

# Radius: linear (constant step) → find where it hits 0
rad_coeffs   = np.polyfit(iters_seg, radii_seg, 1)
converge_iter = int(-rad_coeffs[1] / rad_coeffs[0])

# Lat & lon: deltas decrease linearly → quadratic fit to actual values
lat_coeffs = np.polyfit(iters_seg, lats_seg, 2)
lon_coeffs = np.polyfit(iters_seg, lons_seg, 2)

iters_proj = np.arange(iters_seg[0], converge_iter + 1)
lat_proj   = np.polyval(lat_coeffs, iters_proj)
lon_proj   = np.polyval(lon_coeffs, iters_proj)
rad_proj   = np.polyval(rad_coeffs, iters_proj)

converge_lat = np.polyval(lat_coeffs, converge_iter)
converge_lon = np.polyval(lon_coeffs, converge_iter)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f'Convergence Projection (quadratic lat/lon, linear radius)\n'
    f'Radius → 0 at iteration {converge_iter}  |  '
    f'lat {converge_lat:.5f}°   lon {converge_lon:.5f}°',
    fontsize=11
)

for ax, actual, proj, label, color in zip(
    axes,
    [lats_seg, lons_seg, radii_seg],
    [lat_proj,  lon_proj,  rad_proj],
    ['Latitude (°)', 'Longitude (°)', 'Radius (m)'],
    ['steelblue', 'darkorange', 'seagreen'],
):
    ax.plot(iters_seg, actual, color=color, linewidth=2, label='Actual')
    ax.plot(iters_proj, proj, color=color, linewidth=1.2,
            linestyle='--', alpha=0.6, label='Projected (quadratic)')
    ax.axvline(converge_iter, color='red', linewidth=0.8,
               linestyle=':', label=f'Convergence iter {converge_iter}')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Iteration')
axes[-1].axhline(0, color='red', linewidth=0.8, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

# Also show the delta curves to confirm the linear-decay assumption
fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig2.suptitle('Δ Latitude & Δ Longitude per step (456→503) — linear decay confirms quadratic model')

d_iters = iters_seg[1:]
d_lat = np.diff(lats_seg)
d_lon = np.diff(lons_seg)
dlat_fit = np.polyfit(d_iters, d_lat, 1)
dlon_fit = np.polyfit(d_iters, d_lon, 1)

ax1.plot(d_iters, d_lat, color='steelblue', label='Δ lat actual')
ax1.plot(d_iters, np.polyval(dlat_fit, d_iters), 'k--', linewidth=1, label='linear fit')
ax1.set_ylabel('Δ Latitude'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(d_iters, d_lon, color='darkorange', label='Δ lon actual')
ax2.plot(d_iters, np.polyval(dlon_fit, d_iters), 'k--', linewidth=1, label='linear fit')
ax2.set_ylabel('Δ Longitude'); ax2.set_xlabel('Iteration')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Convergence iteration : {converge_iter}")
print(f"Converged latitude    : {converge_lat:.6f}°")
print(f"Converged longitude   : {converge_lon:.6f}°")


In [ ]:
import folium

clat = (max(lats) + min(lats)) / 2
clon = (max(lons) + min(lons)) / 2

m = folium.Map(location=[clat, clon], zoom_start=3, tiles='OpenStreetMap')

# Path of circle centers
folium.PolyLine(list(zip(lats, lons)), color='navy', weight=1.5, opacity=0.7).add_to(m)

# Sample circles every 50 iterations, colour fades green → red
for i in range(0, len(lats), 50):
    frac = i / (len(lats) - 1)
    r = int(220 * frac)
    g = int(180 * (1 - frac))
    color = f'#{r:02x}{g:02x}50'
    folium.Circle(
        location=[lats[i], lons[i]],
        radius=radii[i],
        color=color,
        fill=False,
        weight=1.5,
        popup=f'Iter {i+1} | radius {radii[i]/1000:.0f} km',
    ).add_to(m)

folium.Marker([lats[0],  lons[0]],  popup='Start (iter 1)',         icon=folium.Icon(color='green')).add_to(m)
folium.Marker([lats[-1], lons[-1]], popup=f'End (iter {len(lats)})', icon=folium.Icon(color='red')).add_to(m)

m.save('circle_map.html')
m


In [ ]:
import math
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Ellipse
from IPython.display import HTML

STEP = 5  # animate every Nth iteration (lower = smoother but slower to render)

fig, ax = plt.subplots(figsize=(11, 8))
ax.set_xlim(min(lons) - 3, max(lons) + 3)
ax.set_ylim(min(lats) - 3, max(lats) + 3)
ax.set_aspect('equal')
ax.set_facecolor('#cce0f0')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.25)

trail_line,  = ax.plot([], [], '-',  color='navy',  linewidth=0.8, alpha=0.5)
center_dot,  = ax.plot([], [], 'o',  color='red',   markersize=4)
circle_patch = Ellipse((lons[0], lats[0]), 0, 0,
                        fill=False, edgecolor='crimson', linewidth=1.5)
ax.add_patch(circle_patch)
title = ax.set_title('')

def animate(frame):
    i = min(frame * STEP, len(lats) - 1)
    lat, lon, r = lats[i], lons[i], radii[i]

    r_lat = r / 111_320
    r_lon = r / (111_320 * math.cos(math.radians(lat)))

    circle_patch.set_center((lon, lat))
    circle_patch.set_width(2 * r_lon)
    circle_patch.set_height(2 * r_lat)
    trail_line.set_data(lons[:i + 1], lats[:i + 1])
    center_dot.set_data([lon], [lat])
    title.set_text(
        f'Iteration {i + 1}  |  '
        f'lat {lat:.3f}°   lon {lon:.3f}°   '
        f'radius {r / 1000:.0f} km'
    )
    return circle_patch, trail_line, center_dot, title

n_frames = math.ceil(len(lats) / STEP)
anim = FuncAnimation(fig, animate, frames=n_frames, interval=80, blit=True)
plt.close()
HTML(anim.to_jshtml())


In [ ]:
import requests
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import HTML

# Arizona bounding box
LAT_MIN, LAT_MAX = 31.33, 37.00
LON_MIN, LON_MAX = -114.82, -109.05
STEP_DEG = 0.5          # grid resolution in degrees
DATE     = "2026-05-22"
TZID     = "America/Phoenix"

# Build grid
lat_pts = np.arange(LAT_MIN, LAT_MAX + STEP_DEG, STEP_DEG)
lon_pts = np.arange(LON_MIN, LON_MAX + STEP_DEG, STEP_DEG)
grid    = [(round(lat, 4), round(lon, 4)) for lat in lat_pts for lon in lon_pts]
print(f"Querying {len(grid)} points over Arizona ({STEP_DEG}° grid) for {DATE}…")

def fetch_sun(lat, lon):
    url = "https://api.sunrise-sunset.org/json"
    r = requests.get(url, params={
        "lat":       lat,
        "lng":       lon,
        "date":      DATE,
        "formatted": 0,
        "tzid":      TZID,
    }, timeout=10)
    r.raise_for_status()
    res = r.json()["results"]
    return {
        "lat":                       lat,
        "lon":                       lon,
        "sunrise":                   res["sunrise"],
        "sunset":                    res["sunset"],
        "solar_noon":                res["solar_noon"],
        "day_length_sec":            res["day_length"],
        "civil_twilight_begin":      res["civil_twilight_begin"],
        "civil_twilight_end":        res["civil_twilight_end"],
        "nautical_twilight_begin":   res["nautical_twilight_begin"],
        "nautical_twilight_end":     res["nautical_twilight_end"],
        "astronomical_twilight_begin": res["astronomical_twilight_begin"],
        "astronomical_twilight_end":   res["astronomical_twilight_end"],
    }

rows = []
errors = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(fetch_sun, lat, lon): (lat, lon) for lat, lon in grid}
    for i, future in enumerate(as_completed(futures), 1):
        lat, lon = futures[future]
        try:
            rows.append(future.result())
        except Exception as e:
            errors.append((lat, lon, str(e)))
        if i % 20 == 0 or i == len(grid):
            print(f"  {i}/{len(grid)} done")

sun_df = pd.DataFrame(rows).sort_values(["lat", "lon"]).reset_index(drop=True)

# Add day_length in hours for readability
sun_df["day_length_hr"] = (sun_df["day_length_sec"] / 3600).round(4)

sun_df.to_csv("arizona_sunrise_sunset.csv", index=False)
print(f"\nSaved {len(sun_df)} rows → arizona_sunrise_sunset.csv")
if errors:
    print(f"Errors: {errors}")

html = f'<div style="height:500px;overflow:auto;">{sun_df.to_html(index=False)}</div>'
display(HTML(html))
